Version control: 2026-10-14 - 18:30
Luis Anunciacao, PhD

# DA - AGEM Time Series

Loads `df.rds` from the DP notebook. Nothing is transformed here.

**Tests.** Two-sided Wilcoxon rank-sum (Mann-Whitney U) tests between consecutive waves, by company, for scale items. Waves are independent samples: SurveyMonkey collected no respondent ID, so answers cannot be paired. Lideres T2 items 6 and 20 are reverse-coded for the tests only (DP §2); counts are raw. "—" = item not applied in that wave.

In [1]:
pacman::p_load(tidyverse)

options(repr.matrix.max.rows = 400, repr.matrix.max.cols = 20)

# DP output from GitHub (Processing and analysis/Importable data/df.rds); private repository, token from the git credential store (gitcreds)
df <- readRDS(gzcon(curl::curl("https://raw.githubusercontent.com/anovabr/AGEM-Time-Series/main/Processing%20and%20analysis/Importable%20data/df.rds",
                               handle = curl::new_handle() |> curl::handle_setheaders(Authorization = paste("token", gitcreds::gitcreds_get()$password)))))

# item x response frequency table by company and wave, with W and p of the rank-sum test (T1 vs T2, T2 vs T3) on the first row of each item
tabela <- function(inst) {
  d <- filter(df, instrumento == inst)
  attr(df, "opcoes") |>
    filter(instrumento == inst) |>
    inner_join(distinct(d, item, empresa, onda), by = "item", relationship = "many-to-many") |>
    left_join(count(d, item, codigo, empresa, onda), by = c("item", "codigo", "empresa", "onda")) |>
    mutate(n = as.character(replace_na(n, 0L))) |>
    bind_rows(d |>
                filter(!is.na(valor_teste)) |>
                reframe(map_dfr(list(c("T1", "T2"), c("T2", "T3")), \(w) if (all(w %in% onda))
                          tibble(par = paste0(w[1], "\u2192", w[2]),
                                 n = with(suppressWarnings(wilcox.test(valor_teste[onda == w[1]], valor_teste[onda == w[2]])), sprintf("W = %.1f; p = %.3f", statistic, p.value)))),
                        .by = c(item, empresa)) |>
                transmute(item, empresa, onda = par, n, codigo = 1L)) |>
    pivot_wider(id_cols = c(item, codigo), names_from = c(empresa, onda), values_from = n, names_sort = TRUE) |>
    arrange(item, codigo) |>
    left_join(distinct(d, item, item_texto), by = "item") |>
    left_join(filter(attr(df, "opcoes"), instrumento == inst) |> select(item, codigo, resposta), by = c("item", "codigo")) |>
    mutate(Item = if_else(codigo == 1, paste0(item, ". ", item_texto), ""), Resposta = paste0("(", codigo, ") ", resposta),
           across(-c(item, codigo, item_texto, resposta, Item, Resposta), ~ if_else(is.na(.) & (!str_detect(cur_column(), "\u2192") | codigo == 1), "\u2014", replace_na(., "")))) |>
    (\(t) select(t, Item, Resposta, all_of(sort(setdiff(names(t), c("item", "codigo", "item_texto", "resposta", "Item", "Resposta"))))))()
}

## §1 Table 1 - Inventário de Saúde Mental exclusivo para Líderes

In [2]:
tabela("lideres")

Item,Resposta,finx_T1,finx_T1→T2,finx_T2,finx_T2→T3,finx_T3,xforce_T1,xforce_T1→T2,xforce_T2,xforce_T2→T3,xforce_T3
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1. sinto que a carga de responsabilidade pelas decisoes e elevada.,(1) discordo totalmente,3,W = 90.5; p = 0.237,4,W = 94.5; p = 0.305,2,1,W = 19.5; p = 0.870,2,W = 12.5; p = 0.416,2
,(2) discordo parcialmente,5,,0,,2,1,,1,,0
,(3) nem concordo nem discordo,5,,5,,3,2,,1,,1
,(4) concordo parcialmente,0,,7,,4,2,,1,,0
,(5) concordo totalmente,2,,0,,4,0,,1,,0
2. tenho suporte adequado = (recursos/informacao) para tomar decisoes de forma segura.,(1) discordo totalmente,0,W = 127.5; p = 0.766,0,W = 128.0; p = 0.752,1,0,W = 20.0; p = 0.788,0,W = 4.0; p = 0.167,0
,(2) discordo parcialmente,2,,2,,1,1,,1,,0
,(3) nem concordo nem discordo,2,,2,,1,1,,1,,0
,(4) concordo parcialmente,3,,5,,7,3,,4,,2


## §2 Table 2 - Inventário de Saúde Emocional

In [3]:
tabela("emocional")

Item,Resposta,finx_T1,finx_T1→T2,finx_T2,xforce_T1,xforce_T1→T2,xforce_T2
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1. consegue lidar bem com frustracoes e imprevistos? (ex.: manter a calma mesmo quando nao cumpre um prazo),(1) nunca,0,W = 3190.5; p = 0.091,0,1,W = 228.5; p = 0.822,0
,(2) raramente,12,,4,1,,0
,(3) as vezes,28,,27,7,,7
,(4) muitas vezes,39,,46,14,,10
,(5) sempre,7,,9,2,,2
2. mantem o autocontrole em situacoes de estresse ou conflito? (ex.: nao perder a cabeca em discussoes),(1) nunca,0,W = 4045.0; p = 0.246,0,0,W = 274.5; p = 0.350,0
,(2) raramente,3,,4,2,,2
,(3) as vezes,23,,24,4,,5
,(4) muitas vezes,41,,48,13,,9
